In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# raw = pd.read_csv('./data/ic2509_250906.csv')
raw = pd.read_csv('./data/ic2509_250912.csv')

In [3]:
klines=raw.loc[:,['time','open','high','low','close','volume']]

In [9]:
from tqsdk.tafunc import ma, ema, abs, std, hhv, llv, count, time_to_datetime, barlast

In [7]:
from tqsdk.ta import MA, EMA

在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


In [22]:
body_thresh=0.5
bsp_n =20
bsp_thresh1= 1.4
bsp_thresh2=2.1

kline=klines[['time','close','open','volume']].copy()
body = kline.close - kline.open
mab = ma(abs(body),bsp_n)  # ma body
# # buy sell pressure body_thresh=0.3 
need_handle = abs(body) < mab*body_thresh
keep_body = abs(body) >= mab*body_thresh
#     对小K线或者十字星 的body 长度进行 标准化
kline.loc[need_handle,'revised_body'] = mab[need_handle]*body_thresh
kline.loc[keep_mab,'revised_body'] = abs(body[keep_body])
bsp=kline.volume/kline.revised_body
ma_bsp = ma(abs(bsp),bsp_n)
kline['ma_bsp'] =ma_bsp
kline['mab'] = mab
kline['bsp'] = bsp
kline['body'] = body
kline['keep_mab'] = keep_mab

bsp_ratio = round(bsp/ma_bsp,4)
kline['bsp_ratio'] = bsp_ratio
#     超轻买卖压  代表大家在观望
light_idx = bsp_ratio < bsp_thresh1
kline.loc[light_idx,'bsp_label'] = 1
# 超重买卖压  代表分歧，可能就是一个阶段的临界值
heavey_idx = (bsp_thresh2 <= bsp_ratio)
kline.loc[heavey_idx,'bsp_label'] = 3
# 普通买卖压
normal_idx = (bsp_thresh1 <= bsp_ratio) & (bsp_ratio < bsp_thresh2)
kline.loc[normal_idx,'bsp_label'] = 2
#     kline=kline.fillna(2)
#     return kline[['bsp_label','bsp_ratio']]
# return kline[['time','body','revised_body','volume','bsp_label','bsp_ratio','ma_bsp','mab','bsp']]
# #     return kline[['bsp_label','revised_body','bsp_ratio']]

In [11]:
kline1 = kline.set_index('time')

In [19]:
mab[need_handle]*body_thresh

22      2.355
25      2.635
28      1.990
30      1.945
33      1.720
        ...  
2084    3.615
2088    3.270
2090    3.195
2093    2.710
2099    3.765
Length: 671, dtype: float64

In [20]:
kline.loc[need_handle,'revised_body'] = mab[need_handle]*body_thresh

In [21]:
kline.loc[need_handle,'revised_body']

22      2.355
25      2.635
28      1.990
30      1.945
33      1.720
        ...  
2084    3.615
2088    3.270
2090    3.195
2093    2.710
2099    3.765
Name: revised_body, Length: 671, dtype: float64

In [18]:
kline.loc[need_handle,'body']

22      0.0
25      2.2
28      1.6
30      0.2
33      1.0
       ... 
2084   -1.8
2088   -1.0
2090    2.4
2093    1.2
2099    2.8
Name: body, Length: 671, dtype: float64

In [17]:
concat(kline.loc[need_handle,'body']

SyntaxError: unexpected EOF while parsing (<ipython-input-17-0443b1df6dc5>, line 1)

In [13]:
kline1.loc['2025/09/11-09:35':'2025/09/11-10:30',:][:10]
# [['body','revised_body','bsp_label','volume','bsp_ratio','ma_bsp','mab','bsp']]

,close,open,volume,revised_body,ma_bsp,mab,bsp,body,keep_mab,bsp_ratio,bsp_label
time,,,,,,,,,,,
2025/09/11-09:35,6841.2,6863.0,5324,5.98,251.914438,5.98,890.301003,-21.8,True,3.5341,3.0
2025/09/11-09:40,6852.4,6842.8,2934,6.18,270.337043,6.18,474.757282,9.6,True,1.7562,2.0
2025/09/11-09:45,6883.6,6853.8,2887,7.41,283.835800,7.41,389.608637,29.8,True,1.3727,1.0
2025/09/11-09:50,6892.8,6885.0,2280,7.74,287.741109,7.74,294.573643,7.8,True,1.0237,1.0
2025/09/11-09:55,6877.8,6892.2,2209,8.35,288.965929,8.35,264.550898,-14.4,True,0.9155,1.0
2025/09/11-10:00,6899.2,6879.2,2188,9.24,291.537800,9.24,236.796537,20.0,True,0.8122,1.0
2025/09/11-10:05,6919.4,6899.0,3536,10.26,296.296164,10.26,344.639376,20.4,True,1.1632,1.0
2025/09/11-10:10,6926.4,6918.0,2168,10.54,295.088017,10.54,205.692600,8.4,True,0.6971,1.0
2025/09/11-10:15,6933.4,6927.0,2288,10.39,297.302020,10.39,220.211742,6.4,True,0.7407,1.0
